In [1]:
import json
from collections import Counter

with open("meged3.json", "r", encoding="utf-8") as f:
    data = json.load(f)

counter = Counter(item["tv_form"] for item in data if item.get("tv_form"))

VALID_CLASSES = [tv for tv, count in counter.items() if count >= 20]

filtered_data = [
    item for item in data
    if item.get("tv_form") in VALID_CLASSES
]

print("Total after filtering:", len(filtered_data))
print("Remaining classes:", len(set(x["tv_form"] for x in filtered_data)))

Total after filtering: 3209
Remaining classes: 16


In [2]:
import json

with open("meged3.json", "r", encoding="utf-8") as f:
    data = json.load(f)

count_double = 0
count_total = 0

for item in data:
    text = item["text"].lower()

    if "өйткені" in text:
        count_total += 1

        if text.count("өйткені") > 1:
            count_double += 1

print("Total өйткені:", count_total)
print("Multiple өйткені:", count_double)

if count_total > 0:
    print("Percent:", round(count_double / count_total * 100, 2), "%")

Total өйткені: 227
Multiple өйткені: 28
Percent: 12.33 %


In [3]:
import json
from collections import Counter

with open("meged3.json", "r", encoding="utf-8") as f:
    data = json.load(f)

marker_counter = Counter()

for item in data:

    tv = item.get("tv_form", "")
    
    # интересующий нас tv form
    if "Tv =fin" in tv:

        for span in item.get("spans", []):
            if "MARKER" in span.get("labels", []):
                marker = span["text"].lower()
                marker_counter[marker] += 1

print("Markers with Tv=fin structure:\n")

for marker, count in marker_counter.most_common():
    print(f"{marker:20} {count}")

Markers with Tv=fin structure:

өйткені              226
себебі               201
сондықтан            201
сол себепті          200
сол үшін             200
неге десең           193
 себебі              1


In [4]:
import json
from collections import defaultdict, Counter

with open("meged3.json", "r", encoding="utf-8") as f:
    data = json.load(f)

tv_marker_map = defaultdict(Counter)

for item in data:

    tv = item.get("tv_form", "UNKNOWN")

    for span in item.get("spans", []):
        if "MARKER" in span.get("labels", []):
            marker = span["text"].lower().strip()
            tv_marker_map[tv][marker] += 1


for tv, markers in tv_marker_map.items():

    print("\n======================================")
    print("TV FORM:", tv)
    print("--------------------------------------")

    for marker, count in markers.most_common():
        print(f"{marker:20} {count}")


TV FORM: Tv=ғандықтан, гендіктен, қандықтан, кендіктен
--------------------------------------
болғандықтан         49
дамығандықтан        6
өзгергендіктен       6
күшейгендіктен       6
қалыптасқандықтан    6
тұрақтанғандықтан    5
кеңейгендіктен       5
алғандықтан          4
тереңдегендіктен     4
отырғандықтан        3
асырылғандықтан      3
сақталғандықтан      3
нақтыланғандықтан    3
жаңарғандықтан       3
нығайғандықтан       3
бекітілгендіктен     3
білгендіктен         2
жүргізілгендіктен    2
білмегендіктен       2
болмағандықтан       2
қалғандықтан         2
алмағандықтан        2
берілгендіктен       2
зерттелмегендіктен   2
өткізгендіктен       2
жүйеленбегендіктен   2
қабылданғандықтан    2
дәлелденгендіктен    2
шиеленіскендіктен    2
ұшырағандықтан       2
реттелгендіктен      2
медиатизацияланғандықтан 2
жетілдірілгендіктен  2
қаралғандықтан       2
өскендіктен          2
жүйеленгендіктен     2
жаңғырғандықтан      2
қолданылғандықтан    2
жатқандықтан         2
бөл

In [5]:
from sklearn.preprocessing import LabelEncoder

texts = [x["text"] for x in filtered_data]
labels = [x["tv_form"] for x in filtered_data]

label_encoder = LabelEncoder()
encoded_labels = label_encoder.fit_transform(labels)

num_labels = len(label_encoder.classes_)
print("Number of classes:", num_labels)
print("Classes:", list(label_encoder.classes_))

id2label = {i: label for i, label in enumerate(label_encoder.classes_)}
label2id = {label: i for i, label in enumerate(label_encoder.classes_)}

Number of classes: 16
Classes: [np.str_('Tv=атын//на, етін//не, йтін//не, йтын//на, итін//не'), np.str_('Tv=атындықтан, етіндіктен, итіндіктен'), np.str_('Tv=май, мей, бай, бей, пей'), np.str_('Tv=п, е, й'), np.str_('Tv=ған соң, ген соң, қан соң, кен соң'), np.str_('Tv=ған//нан, ген//нен, қан//нан, кен//нен'), np.str_('Tv=ған=//=на, ген=//=не, қан=//=на, кен=//=не'), np.str_('Tv=ғандықтан, гендіктен, қандықтан, кендіктен'), np.str_('Tv=ғаннан кейін, геннен кейін, қаннан кейін, кеннен кейін'), np.str_('Tv=ғаны үшін, гені үшін, кені үшін, қаны үшін'), np.str_('[(N1) Tv =fin] неге десең, неге десеңіз [(N1) Vfin.]'), np.str_('[(N1) Tv =fin] себебі [(N1) Vfin.]'), np.str_('[(N1) Tv =fin] сол себепті [(N1) Vfin.]'), np.str_('[(N1) Tv =fin] сол үшін [(N1) Vfin.]'), np.str_('[(N1) Tv =fin] сондықтан [(N1) Vfin.]'), np.str_('[(N1) Tv =fin] өйткені [(N1) Vfin.]')]


In [6]:
from sklearn.model_selection import train_test_split
from datasets import Dataset

train_texts, test_texts, train_labels, test_labels = train_test_split(
    texts,
    encoded_labels,
    test_size=0.2,
    random_state=42,
    stratify=encoded_labels
)

train_dataset = Dataset.from_dict({
    "text": train_texts,
    "labels": train_labels
})

test_dataset = Dataset.from_dict({
    "text": test_texts,
    "labels": test_labels
})

c:\Users\egisb\voice1\kazFineTune\.venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [7]:
import torch

print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "None")

CUDA available: True
GPU: NVIDIA GeForce RTX 4060 Laptop GPU


In [8]:
from transformers import AutoTokenizer

model_ckpt = "Eraly-ml/KazBERT"
tokenizer = AutoTokenizer.from_pretrained(model_ckpt)

def tokenize(example):
    return tokenizer(
        example["text"],
        truncation=True,
        padding="max_length",
        max_length=128
    )

train_dataset = train_dataset.map(tokenize, batched=True)
test_dataset = test_dataset.map(tokenize, batched=True)

Map: 100%|██████████| 642/642 [00:00<00:00, 14244.08 examples/s]


In [9]:
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained(
    model_ckpt,
    num_labels=num_labels,
    id2label=id2label,
    label2id=label2id
)

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at Eraly-ml/KazBERT and are newly initialized: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight', 'classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [10]:
import numpy as np
from sklearn.metrics import f1_score, accuracy_score

def compute_metrics(p):
    preds = np.argmax(p.predictions, axis=1)
    labels = p.label_ids

    return {
        "accuracy": accuracy_score(labels, preds),
        "macro_f1": f1_score(labels, preds, average="macro")
    }

In [11]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./kazbert_tvform_final",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=8,
    weight_decay=0.01,
    fp16=True,
    load_best_model_at_end=True,
    logging_dir="./logs"
)

In [ ]:
# чтобы проверить другие модели
# from transformers import TrainingArguments

# training_args = TrainingArguments(
#     output_dir="./temp_output",   
#     eval_strategy="epoch",
#     save_strategy="no",         
#     learning_rate=2e-5,
#     per_device_train_batch_size=16,
#     per_device_eval_batch_size=16,
#     num_train_epochs=8,
#     weight_decay=0.01,
#     fp16=True,
#     load_best_model_at_end=False,  
#     logging_dir="./logs",
#     seed=42
# )

In [ ]:
# import torch
# import numpy as np
# import random

# torch.manual_seed(42)
# np.random.seed(42)
# random.seed(42)

In [12]:
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics
)

trainer.train()

C:\Users\egisb\AppData\Local\Temp\ipykernel_4708\4223154140.py:3: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Epoch,Training Loss,Validation Loss,Accuracy,Macro F1
1,No log,0.205135,0.967290,0.966068
2,No log,0.033087,0.993769,0.993611
3,No log,0.014336,0.996885,0.996924
4,0.484600,0.028772,0.993769,0.993753
5,0.484600,0.018085,0.995327,0.995350
6,0.484600,0.008532,0.998442,0.998526
7,0.010000,0.007966,0.998442,0.998526
8,0.010000,0.008243,0.998442,0.998526


TrainOutput(global_step=1288, training_loss=0.19346267160791789, metrics={'train_runtime': 2234.308, 'train_samples_per_second': 9.191, 'train_steps_per_second': 0.576, 'total_flos': 1350981955780608.0, 'train_loss': 0.19346267160791789, 'epoch': 8.0})

In [13]:
results = trainer.evaluate()
print(results)

{'eval_loss': 0.007966133765876293, 'eval_accuracy': 0.9984423676012462, 'eval_macro_f1': 0.9985261478707171, 'eval_runtime': 1.6471, 'eval_samples_per_second': 389.775, 'eval_steps_per_second': 24.892, 'epoch': 8.0}


In [14]:
from sklearn.metrics import classification_report

predictions = trainer.predict(test_dataset)
preds = np.argmax(predictions.predictions, axis=1)

print(classification_report(
    test_labels,
    preds,
    target_names=label_encoder.classes_
))

                                                           precision    recall  f1-score   support

      Tv=атын//на, етін//не, йтін//не, йтын//на, итін//не       1.00      1.00      1.00        39
                    Tv=атындықтан, етіндіктен, итіндіктен       1.00      1.00      1.00        38
                               Tv=май, мей, бай, бей, пей       1.00      1.00      1.00        43
                                               Tv=п, е, й       1.00      1.00      1.00        37
                    Tv=ған соң, ген соң, қан соң, кен соң       1.00      1.00      1.00        40
                Tv=ған//нан, ген//нен, қан//нан, кен//нен       1.00      1.00      1.00        40
            Tv=ған=//=на, ген=//=не, қан=//=на, кен=//=не       1.00      1.00      1.00        41
            Tv=ғандықтан, гендіктен, қандықтан, кендіктен       0.98      1.00      0.99        40
Tv=ғаннан кейін, геннен кейін, қаннан кейін, кеннен кейін       1.00      1.00      1.00        40
         

In [15]:
trainer.save_model("kazbert_tvform_model_final")
tokenizer.save_pretrained("kazbert_tvform_model_final")

('kazbert_tvform_model_final\\tokenizer_config.json',
 'kazbert_tvform_model_final\\special_tokens_map.json',
 'kazbert_tvform_model_final\\vocab.txt',
 'kazbert_tvform_model_final\\added_tokens.json',
 'kazbert_tvform_model_final\\tokenizer.json')

In [16]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification, pipeline

model_dir = "kazbert_tvform_model_final"

tokenizer = AutoTokenizer.from_pretrained(model_dir)
model = AutoModelForSequenceClassification.from_pretrained(model_dir)

tv_pipeline = pipeline(
    "text-classification",
    model=model,
    tokenizer=tokenizer
)

Device set to use cuda:0


In [17]:
sentence = "Қысқа да әрі нақты, көпке түсінікті болғандықтан студенттерге бір деммен оқу қиынға соқпауы мүмкін."
prediction = tv_pipeline(sentence)

print(prediction)

[{'label': 'Tv=ғандықтан, гендіктен, қандықтан, кендіктен', 'score': 0.9955139756202698}]


In [ ]:
# def train_sequence_model(model_ckpt, task_name):

#     tokenizer = AutoTokenizer.from_pretrained(model_ckpt)

#     model = AutoModelForSequenceClassification.from_pretrained(
#         model_ckpt,
#         num_labels=num_labels,
#         id2label=id2label,
#         label2id=label2id
#     )

#     trainer = Trainer(
#         model=model,
#         args=training_args,
#         train_dataset=train_dataset,
#         eval_dataset=test_dataset,
#         tokenizer=tokenizer,
#         compute_metrics=compute_metrics
#     )

#     trainer.train()

#     results = trainer.evaluate()

#     print(f"\n===== {task_name} =====")
#     print(results)

#     return results



In [ ]:
# from transformers import Trainer

# results_kazbert = train_sequence_model("Eraly-ml/KazBERT", "KazBERT")
# results_xlmr = train_sequence_model("xlm-roberta-base", "XLM-R")
# results_mbert = train_sequence_model("bert-base-multilingual-cased", "mBERT")

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at Eraly-ml/KazBERT and are newly initialized: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight', 'classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
C:\Users\egisb\AppData\Local\Temp\ipykernel_28532\2004852314.py:12: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Epoch,Training Loss,Validation Loss,Accuracy,Macro F1
1,No log,0.499287,0.819495,0.779444
2,No log,0.092703,0.989170,0.989285
3,No log,0.063649,0.989170,0.989271
4,No log,0.021791,0.996390,0.996428
5,No log,0.023047,0.996390,0.996428
6,No log,0.023749,0.996390,0.996428
7,No log,0.019451,0.996390,0.996428
8,0.250300,0.019424,0.996390,0.996428



===== KazBERT =====
{'eval_loss': 0.019424259662628174, 'eval_accuracy': 0.9963898916967509, 'eval_macro_f1': 0.9964280133056505, 'eval_runtime': 0.4595, 'eval_samples_per_second': 602.854, 'eval_steps_per_second': 39.175, 'epoch': 8.0}


Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`
Some weights of XLMRobertaForSequenceClassification were not initialized from the model checkpoint at xlm-roberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
C:\Users\egisb\AppData\Local\Temp\ipykernel_28532\2004852314.py:12: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Epoch,Training Loss,Validation Loss,Accuracy,Macro F1
1,No log,1.642951,0.342960,0.240397
2,No log,1.015008,0.649819,0.642572
3,No log,0.669337,0.754513,0.743622
4,No log,0.425896,0.830325,0.821834
5,No log,0.351550,0.877256,0.876791
6,No log,0.267227,0.920578,0.920822
7,No log,0.280180,0.880866,0.875497
8,0.862000,0.217683,0.942238,0.942569



===== XLM-R =====
{'eval_loss': 0.21768327057361603, 'eval_accuracy': 0.9422382671480144, 'eval_macro_f1': 0.9425685425685426, 'eval_runtime': 0.4573, 'eval_samples_per_second': 605.727, 'eval_steps_per_second': 39.361, 'epoch': 8.0}


'(MaxRetryError('HTTPSConnectionPool(host=\'huggingface.co\', port=443): Max retries exceeded with url: /bert-base-multilingual-cased/resolve/main/tokenizer_config.json (Caused by NameResolutionError("HTTPSConnection(host=\'huggingface.co\', port=443): Failed to resolve \'huggingface.co\' ([Errno 11001] getaddrinfo failed)"))'), '(Request ID: 72f057d1-7d87-4834-912f-b7d2b9c8ef3b)')' thrown while requesting HEAD https://huggingface.co/bert-base-multilingual-cased/resolve/main/tokenizer_config.json
Retrying in 1s [Retry 1/5].
Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`
Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-multilingual-cased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a dow

Epoch,Training Loss,Validation Loss,Accuracy,Macro F1
1,No log,1.281153,0.635379,0.614574
2,No log,0.701135,0.722022,0.703654
3,No log,0.432952,0.830325,0.832337
4,No log,0.319724,0.851986,0.848391
5,No log,0.222620,0.913357,0.915025
6,No log,0.200551,0.924188,0.924114
7,No log,0.170789,0.945848,0.946110
8,0.636700,0.142518,0.945848,0.946209



===== mBERT =====
{'eval_loss': 0.14251770079135895, 'eval_accuracy': 0.9458483754512635, 'eval_macro_f1': 0.9462090413527264, 'eval_runtime': 0.9224, 'eval_samples_per_second': 300.301, 'eval_steps_per_second': 19.514, 'epoch': 8.0}
